# Dashboard Data Preparation

This notebook prepares final Dataset V3 outputs for dashboard integration.

It combines hotspot detection, intervention simulation, model evaluation, and explainability outputs into dashboard-ready datasets. These files can be used by the Streamlit dashboard without retraining the model.

In [2]:
import pandas as pd
import numpy as np
import os

# Create dashboard output folder
os.makedirs("../outputs/dashboard_data", exist_ok=True)

# Load final pipeline outputs
hotspot_predictions = pd.read_csv(
    "../data/processed/hotspot_predictions_v3.csv"
)

scenario_predictions = pd.read_csv(
    "../data/processed/scenario_predictions_v3.csv"
)

hotspot_zones = pd.read_csv(
    "../outputs/reports/hotspot_zone_priority_v3.csv"
)

intervention_zones = pd.read_csv(
    "../outputs/reports/intervention_priority_v3.csv"
)

model_comparison = pd.read_csv(
    "../outputs/reports/v2_vs_v3_xgboost_comparison.csv"
)

shap_importance = pd.read_csv(
    "../outputs/reports/shap_feature_importance_v3.csv"
)

scenario_summary = pd.read_csv(
    "../outputs/reports/scenario_summary_v3.csv"
)

print("All final pipeline outputs loaded successfully.")
print("Hotspot predictions:", hotspot_predictions.shape)
print("Scenario predictions:", scenario_predictions.shape)
print("Hotspot zones:", hotspot_zones.shape)
print("Intervention zones:", intervention_zones.shape)
print("Model comparison:", model_comparison.shape)
print("SHAP importance:", shap_importance.shape)
print("Scenario summary:", scenario_summary.shape)

All final pipeline outputs loaded successfully.
Hotspot predictions: (9893, 21)
Scenario predictions: (9893, 28)
Hotspot zones: (334, 13)
Intervention zones: (334, 10)
Model comparison: (2, 6)
SHAP importance: (11, 3)
Scenario summary: (2, 3)


In [2]:
# Keep only dashboard-relevant columns for the point-level heat map
dashboard_hotspot_map = hotspot_predictions[
    [
        "Latitude",
        "Longitude",
        "LST",
        "Predicted_LST",
        "Heat_Category",
        "NDVI",
        "NDBI",
        "Elevation",
        "Population",
        "LandCover_Class",
        "Zone_ID"
    ]
].copy()

dashboard_hotspot_map = dashboard_hotspot_map.rename(
    columns={
        "LST": "Observed_LST_C",
        "Predicted_LST": "Predicted_LST_C"
    }
)

print("Dashboard hotspot map data:")
display(dashboard_hotspot_map.head())
print("Shape:", dashboard_hotspot_map.shape)

Dashboard hotspot map data:


,Latitude,Longitude,Observed_LST_C,Predicted_LST_C,Heat_Category,NDVI,NDBI,Elevation,Population,LandCover_Class,Zone_ID
0,19.970832,75.438700,43.919170,44.202328,High,0.162286,0.127207,919,1.562392,Cropland,12_14
1,19.729093,75.209548,39.060455,39.296585,Low,0.570751,-0.272423,488,1.361367,Cropland,0_3
2,19.788775,75.225639,40.231127,43.298996,Low,0.099371,0.055877,505,15.386780,Grassland,3_4
3,19.960297,75.544582,44.884761,45.497520,High,0.196037,0.121404,757,2.643492,Cropland,11_20
4,19.867709,75.486062,48.502735,47.015200,Very High,0.165262,0.133151,586,31.397209,Grassland,7_17


Shape: (9893, 11)


In [3]:
# Keep scenario information needed for intervention maps and recommendations
dashboard_scenario_map = scenario_predictions[
    [
        "Latitude",
        "Longitude",
        "Zone_ID",
        "Baseline_Predicted_LST",
        "Greening_Predicted_LST",
        "Cool_Surface_Predicted_LST",
        "Greening_Cooling_C",
        "Cool_Surface_Cooling_C",
        "Best_Cooling_C",
        "Recommended_Intervention",
        "Population",
        "Heat_Category"
    ]
].copy()

dashboard_scenario_map = dashboard_scenario_map.rename(
    columns={
        "Baseline_Predicted_LST": "Baseline_Predicted_LST_C",
        "Greening_Predicted_LST": "Greening_Predicted_LST_C",
        "Cool_Surface_Predicted_LST": "Cool_Surface_Predicted_LST_C"
    }
)

print("Dashboard scenario map data:")
display(dashboard_scenario_map.head())
print("Shape:", dashboard_scenario_map.shape)

Dashboard scenario map data:


,Latitude,Longitude,Zone_ID,Baseline_Predicted_LST_C,Greening_Predicted_LST_C,Cool_Surface_Predicted_LST_C,Greening_Cooling_C,Cool_Surface_Cooling_C,Best_Cooling_C,Recommended_Intervention,Population,Heat_Category
0,19.970832,75.438700,12_14,44.202328,44.448135,44.449990,-0.245808,-0.247662,-0.245808,Urban greening,1.562392,High
1,19.729093,75.209548,0_3,39.296585,39.301758,39.708134,-0.005173,-0.411549,-0.005173,Urban greening,1.361367,Low
2,19.788775,75.225639,3_4,43.298996,43.949340,43.719433,-0.650345,-0.420437,-0.420437,Cool/permeable surfaces,15.386780,Low
3,19.960297,75.544582,11_20,45.497520,45.407787,45.506027,0.089733,-0.008507,0.089733,Urban greening,2.643492,High
4,19.867709,75.486062,7_17,47.015200,46.217350,46.364570,0.797852,0.650631,0.797852,Urban greening,31.397209,Very High


Shape: (9893, 12)


In [4]:
# Keep the top 10 hotspot zones for a dashboard priority table
dashboard_priority_zones = hotspot_zones[
    hotspot_zones["Hotspot_Priority"] == "High Priority"
].copy()

dashboard_priority_zones = dashboard_priority_zones.sort_values(
    by="Mean_LST",
    ascending=False
).reset_index(drop=True)

dashboard_priority_zones.insert(
    0,
    "Priority_Rank",
    range(1, len(dashboard_priority_zones) + 1)
)

print("Dashboard priority zones:")
display(dashboard_priority_zones)

Dashboard priority zones:


,Priority_Rank,Zone_ID,Mean_LST,Mean_Predicted_LST,Max_LST,Mean_NDVI,Mean_NDBI,Mean_Population,Point_Count,Very_High_Hotspot_Count,Zone_Latitude,Zone_Longitude,Hotspot_Percentage,Hotspot_Priority
0,1,4_22,48.158655,46.650745,50.447589,0.222963,0.092786,2.584655,6,5,19.810988,75.584888,83.33,High Priority
1,2,4_21,47.612930,46.940758,49.481998,0.188821,0.103167,2.618992,29,27,19.813187,75.571824,93.10,High Priority
2,3,5_20,47.552526,45.852337,50.034008,0.202059,0.047122,2.840524,38,35,19.832173,75.553111,92.11,High Priority
3,4,3_21,47.517681,46.191303,50.257889,0.204836,0.075184,2.340091,18,14,19.799522,75.569608,77.78,High Priority
4,5,4_20,47.516779,46.277410,50.608236,0.197170,0.058913,2.684914,36,31,19.812505,75.552733,86.11,High Priority
5,6,0_15,47.465366,47.156380,47.465366,0.127480,0.101048,2.911846,1,1,19.741849,75.445563,100.00,High Priority
6,7,6_20,47.432254,45.721590,50.309159,0.194545,0.076918,5.377040,32,29,19.853783,75.551956,90.62,High Priority
7,8,5_22,47.203155,46.952824,48.760796,0.183787,0.101800,2.367950,7,5,19.828271,75.583563,71.43,High Priority
8,9,7_15,47.186326,46.210580,50.283524,0.178169,0.071149,6.307922,29,23,19.874958,75.452508,79.31,High Priority
9,10,5_21,47.054278,46.454086,50.647543,0.166336,0.113842,2.370612,24,18,19.832870,75.572410,75.00,High Priority


In [5]:
# Keep the top 10 zones with the highest model-based cooling potential
dashboard_intervention_zones = intervention_zones.head(10).copy()

dashboard_intervention_zones.insert(
    0,
    "Cooling_Priority_Rank",
    range(1, len(dashboard_intervention_zones) + 1)
)

print("Dashboard intervention zones:")
display(dashboard_intervention_zones)

Dashboard intervention zones:


,Cooling_Priority_Rank,Zone_ID,Mean_Baseline_LST,Mean_Greening_Cooling_C,Mean_Cool_Surface_Cooling_C,Mean_Best_Cooling_C,Mean_Population,Zone_Latitude,Zone_Longitude,Point_Count,Recommended_Intervention
0,1,0_6,43.506454,1.097858,0.472767,1.169778,2.641625,19.733412,75.272997,30,Urban greening
1,2,13_9,44.616367,0.900969,0.319292,1.034609,1.135171,19.993966,75.332386,30,Urban greening
2,3,6_20,45.721590,0.910720,0.299620,0.922871,5.377040,19.853783,75.551956,32,Urban greening
3,4,9_7,44.981224,0.858206,0.319060,0.885526,14.635487,19.914050,75.290979,27,Urban greening
4,5,4_21,46.940758,0.742268,0.324992,0.796997,2.618992,19.813187,75.571824,29,Urban greening
5,6,4_20,46.277410,0.686613,0.386653,0.791900,2.684914,19.812505,75.552733,36,Urban greening
6,7,2_11,45.697147,0.737511,0.482929,0.791033,1.515464,19.772852,75.371829,34,Urban greening
7,8,5_0,42.711670,0.370827,0.785172,0.785172,1.565967,19.841369,75.161099,1,Cool/permeable surfaces
8,9,10_22,46.160492,0.777173,0.177707,0.777173,2.422858,19.937127,75.583262,4,Urban greening
9,10,13_7,44.584920,0.565610,0.350590,0.774927,1.111110,19.992573,75.290690,32,Urban greening


In [6]:
# Keep readable SHAP importance information for dashboard charts
dashboard_shap = shap_importance[
    ["Feature", "Readable_Feature", "Mean_Absolute_SHAP"]
].copy()

dashboard_shap = dashboard_shap.sort_values(
    by="Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

print("Dashboard SHAP feature importance:")
display(dashboard_shap)

Dashboard SHAP feature importance:


,Feature,Readable_Feature,Mean_Absolute_SHAP
0,NDBI,NDBI,1.281032
1,Elevation,Elevation,0.580169
2,NDVI,NDVI,0.534977
3,Population,Population,0.376957
4,LandCover_Grassland,Land cover: Grassland,0.320300
5,LandCover_Cropland,Land cover: Cropland,0.103129
6,LandCover_Tree cover,LandCover_Tree cover,0.100840
7,LandCover_Shrubland,Land cover: Shrubland,0.040502
8,LandCover_Built-up land,LandCover_Built-up land,0.036793
9,LandCover_Permanent_water_bodies,Land cover: Permanent water bodies,0.029068


In [7]:
# Create a small summary table for dashboard metric cards
v3_result = model_comparison[
    model_comparison["Dataset"] == "Dataset V3"
].iloc[0]

dashboard_kpis = pd.DataFrame({
    "Metric": [
        "Dataset V3 XGBoost R2",
        "Dataset V3 XGBoost MAE (C)",
        "Dataset V3 XGBoost RMSE (C)",
        "Total Sampled Locations",
        "High Priority Hotspot Zones",
        "Mean Urban Greening Cooling (C)",
        "Mean Cool/Permeable Surface Cooling (C)"
    ],
    "Value": [
        v3_result["R2"],
        v3_result["MAE_C"],
        v3_result["RMSE_C"],
        len(hotspot_predictions),
        len(dashboard_priority_zones),
        scenario_summary.loc[
            scenario_summary["Scenario"] == "Urban greening",
            "Mean_Cooling_C"
        ].iloc[0],
        scenario_summary.loc[
            scenario_summary["Scenario"] == "Cool/permeable surfaces",
            "Mean_Cooling_C"
        ].iloc[0]
    ]
})

dashboard_kpis

,Metric,Value
0,Dataset V3 XGBoost R2,0.5691
1,Dataset V3 XGBoost MAE (C),1.6645
2,Dataset V3 XGBoost RMSE (C),2.1163
3,Total Sampled Locations,9893.0000
4,High Priority Hotspot Zones,10.0000
5,Mean Urban Greening Cooling (C),0.2570
6,Mean Cool/Permeable Surface Cooling (C),0.0760


In [8]:
dashboard_hotspot_map.to_csv(
    "../outputs/dashboard_data/dashboard_hotspot_map_v3.csv",
    index=False
)

dashboard_scenario_map.to_csv(
    "../outputs/dashboard_data/dashboard_scenario_map_v3.csv",
    index=False
)

dashboard_priority_zones.to_csv(
    "../outputs/dashboard_data/dashboard_priority_zones_v3.csv",
    index=False
)

dashboard_intervention_zones.to_csv(
    "../outputs/dashboard_data/dashboard_intervention_zones_v3.csv",
    index=False
)

dashboard_shap.to_csv(
    "../outputs/dashboard_data/dashboard_shap_importance_v3.csv",
    index=False
)

model_comparison.to_csv(
    "../outputs/dashboard_data/dashboard_model_comparison.csv",
    index=False
)

scenario_summary.to_csv(
    "../outputs/dashboard_data/dashboard_scenario_summary_v3.csv",
    index=False
)

dashboard_kpis.to_csv(
    "../outputs/dashboard_data/dashboard_kpis_v3.csv",
    index=False
)

print("Dashboard-ready files saved successfully:")
print("outputs/dashboard_data/dashboard_hotspot_map_v3.csv")
print("outputs/dashboard_data/dashboard_scenario_map_v3.csv")
print("outputs/dashboard_data/dashboard_priority_zones_v3.csv")
print("outputs/dashboard_data/dashboard_intervention_zones_v3.csv")
print("outputs/dashboard_data/dashboard_shap_importance_v3.csv")
print("outputs/dashboard_data/dashboard_model_comparison.csv")
print("outputs/dashboard_data/dashboard_scenario_summary_v3.csv")
print("outputs/dashboard_data/dashboard_kpis_v3.csv")

Dashboard-ready files saved successfully:
outputs/dashboard_data/dashboard_hotspot_map_v3.csv
outputs/dashboard_data/dashboard_scenario_map_v3.csv
outputs/dashboard_data/dashboard_priority_zones_v3.csv
outputs/dashboard_data/dashboard_intervention_zones_v3.csv
outputs/dashboard_data/dashboard_shap_importance_v3.csv
outputs/dashboard_data/dashboard_model_comparison.csv
outputs/dashboard_data/dashboard_scenario_summary_v3.csv
outputs/dashboard_data/dashboard_kpis_v3.csv


## Dashboard Data Preparation Conclusion

Final Dataset V3 outputs were converted into dashboard-ready CSV files.

The prepared files include point-level hotspot data, scenario cooling estimates, high-priority hotspot zones, intervention-priority zones, model-comparison metrics, SHAP feature importance, scenario summaries, and KPI values.

These files can now be used directly in a Streamlit dashboard without rerunning model training, hotspot detection, SHAP analysis, or scenario simulation.

In [3]:
# Load final vulnerability outputs
vulnerability_predictions = pd.read_csv(
    "../data/processed/vulnerability_predictions_v3.csv"
)

vulnerability_zones = pd.read_csv(
    "../outputs/reports/zone_vulnerability_v3.csv"
)

print("Vulnerability point data:", vulnerability_predictions.shape)
print("Vulnerability zones:", vulnerability_zones.shape)

Vulnerability point data: (9893, 32)
Vulnerability zones: (334, 11)


In [4]:
dashboard_vulnerability_map = vulnerability_predictions[
    [
        "Latitude",
        "Longitude",
        "Zone_ID",
        "Baseline_Predicted_LST",
        "Population",
        "Heat_Hazard_Score",
        "Population_Exposure_Score",
        "Heat_Vulnerability_Index",
        "Vulnerability_Category"
    ]
].copy()

dashboard_vulnerability_map = dashboard_vulnerability_map.rename(
    columns={
        "Baseline_Predicted_LST": "Baseline_Predicted_LST_C"
    }
)

print("Dashboard vulnerability map data:")
display(dashboard_vulnerability_map.head())

Dashboard vulnerability map data:


,Latitude,Longitude,Zone_ID,Baseline_Predicted_LST_C,Population,Heat_Hazard_Score,Population_Exposure_Score,Heat_Vulnerability_Index,Vulnerability_Category
0,19.970832,75.438700,12_14,44.202328,1.562392,0.741986,0.003887,0.372937,Moderate
1,19.729093,75.209548,0_3,39.296585,1.361367,0.520915,0.003014,0.261964,Low
2,19.788775,75.225639,3_4,43.298996,15.386780,0.701279,0.063965,0.382622,High
3,19.960297,75.544582,11_20,45.497520,2.643492,0.800353,0.008586,0.404469,High
4,19.867709,75.486062,7_17,47.015200,31.397209,0.868745,0.133544,0.501144,Very High


In [5]:
dashboard_vulnerability_zones = vulnerability_zones[
    vulnerability_zones["Vulnerability_Priority"] == "High Priority"
].copy()

dashboard_vulnerability_zones = dashboard_vulnerability_zones.sort_values(
    by="Mean_Heat_Vulnerability_Index",
    ascending=False
).reset_index(drop=True)

print("Dashboard high-priority vulnerability zones:")
display(dashboard_vulnerability_zones)

Dashboard high-priority vulnerability zones:


,Vulnerability_Rank,Zone_ID,Mean_Baseline_LST,Mean_Population,Mean_Heat_Hazard_Score,Mean_Population_Exposure_Score,Mean_Heat_Vulnerability_Index,Zone_Latitude,Zone_Longitude,Point_Count,Vulnerability_Priority
0,1,7_9,41.775953,176.593178,0.632645,0.764538,0.698591,19.873852,75.332506,33,High Priority
1,2,7_11,41.738128,166.682542,0.630940,0.721468,0.676204,19.873805,75.372631,30,High Priority
2,3,7_10,41.376177,168.289624,0.614629,0.728452,0.671541,19.873351,75.352669,34,High Priority
3,4,8_10,41.804171,158.789687,0.633916,0.687167,0.660542,19.893668,75.351290,37,High Priority
4,5,6_10,42.258778,132.779434,0.654403,0.574131,0.614267,19.853567,75.350470,40,High Priority
5,6,9_10,42.105616,127.969677,0.647501,0.553229,0.600365,19.912571,75.354301,30,High Priority
6,7,8_9,41.618842,129.075473,0.625565,0.558035,0.591800,19.894061,75.329542,40,High Priority
7,8,8_8,43.284214,105.591844,0.700613,0.455979,0.578296,19.894664,75.312489,23,High Priority
8,9,8_11,43.220870,104.735721,0.697758,0.452259,0.575008,19.894265,75.373587,30,High Priority
9,10,6_9,43.035631,100.174659,0.689411,0.432437,0.560924,19.852109,75.331116,32,High Priority


In [6]:
dashboard_vulnerability_map.to_csv(
    "../outputs/dashboard_data/dashboard_vulnerability_map_v3.csv",
    index=False
)

dashboard_vulnerability_zones.to_csv(
    "../outputs/dashboard_data/dashboard_vulnerability_zones_v3.csv",
    index=False
)

print("Saved:")
print("outputs/dashboard_data/dashboard_vulnerability_map_v3.csv")
print("outputs/dashboard_data/dashboard_vulnerability_zones_v3.csv")

Saved:
outputs/dashboard_data/dashboard_vulnerability_map_v3.csv
outputs/dashboard_data/dashboard_vulnerability_zones_v3.csv


## Vulnerability Dashboard Update

Dataset V3 heat-vulnerability outputs were added to the dashboard-ready data folder.

The dashboard can now show point-level vulnerability, heat hazard, population exposure, and the top high-priority vulnerability zones.